In [ ]:
import pandas as pd
from tqdm import tqdm
from google import genai
from openai import OpenAI
from cerebras.cloud.sdk import Cerebras

In [ ]:
df = pd.read_csv("5_Responses.csv")

In [ ]:
GEMINI_API_KEY = "AQ.Ab8RN6IQ9GJnpvmk-dHiZ_GP2_54alYxVYVwZcIGfkhl44-3sQ"
CEREBRAS_API_KEY = "csk-6yer4489fnjtt8k8myw5rdfvde8dw9rxvvh2e946jepr26tw"
# GROQ_API_KEY = ""
# OPENROUTER_API_KEY = ""
# OPENAI_API_KEY = ""

gemini_client = genai.Client(api_key=GEMINI_API_KEY)
cerebras_client = Cerebras(api_key=CEREBRAS_API_KEY)
#openai_client = OpenAI(api_key=OPENAI_API_KEY)

In [ ]:
def generate_gemini(prompt):
    try:
        response = gemini_client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt,
        )
        return response.text.strip()

    except Exception as e:
        print(e)
        return ""

In [ ]:
def generate_cerebras(prompt):
    try:
        response = cerebras_client.chat.completions.create(
            model="gemma-4-31b",
            messages=[
                {"role": "user", "content": prompt}
            ]
        )

        return response.choices[0].message.content.strip()

    except Exception as e:
        print(e)
        return ""

In [ ]:
def generate_openai(prompt):
    try:
        response = openai_client.responses.create(
            model="gpt-5-mini",
            input=prompt,
        )

        return response.output_text.strip()

    except Exception as e:
        print(e)
        return ""

---

In [ ]:
LLM = "cerebras"      
START = 0
END = 10           # Generates rows START to END-1   ## 0-10, 10-20, 20-30

In [ ]:
responses = []

for prompt in tqdm(df["prompt"][START:END]):
    if LLM == "gemini":
        responses.append(generate_gemini(prompt))
    elif LLM == "openai":
        responses.append(generate_openai(prompt))
    elif LLM == "cerebras":
        responses.append(generate_cerebras(prompt))

df.loc[START:END-1, f"{LLM}_response"] = responses

df.to_csv("5_Responses.csv", index=False)

-----

models = cerebras_client.models.list()

ModelListResponse(data=[Data(id='zai-glm-4.7', created=0, object='model', owned_by='Cerebras'), Data(id='gpt-oss-120b', created=0, object='model', owned_by='Cerebras'), Data(id='gemma-4-31b', created=0, object='model', owned_by='Cerebras')], object='list')